<a href="https://colab.research.google.com/github/roughhawkbit/digi-inno-road-prod/blob/main/analysis/BART_for_ZeroShotClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Run this notebook on a GPU runtime (e.g., T4 GPU in Google Colab)

# Setup

In [1]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [2]:
import math
import os
import pandas
import sys
from transformers import pipeline

In [3]:
if IN_COLAB:
  dirpath = '/content/digi-inno-road-prod'
  if not os.path.isdir(dirpath):
    # TODO git pull
    !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git
  sys.path.insert(0,dirpath)
else:
  module_path = os.path.abspath(os.path.join('..'))
  if not module_path in sys.path:
      sys.path.insert(0, module_path)

Cloning into 'digi-inno-road-prod'...
remote: Enumerating objects: 697, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 697 (delta 73), reused 14 (delta 7), pack-reused 584 (from 1)
Receiving objects: 100% (697/697), 1.97 MiB | 5.10 MiB/s, done.
Resolving deltas: 100% (438/438), done.


In [4]:
from innoprod.sheet_tools import get_sheet_dfs
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps

# Notebook settings

In [5]:
qual_cols = [
    'Summary review of Edge Digital diagnostic report & current state and key improvement areas',
    'What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?',
    'Details of any existing Digital Strategy',
    'Level of current Strategic Digital Skills/knowledge in the business',
    'Level of current Technical Digital Skills/knowledge in the business',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Summary of the identified problems, including Gap Analysis'
]

drs_col = 'Current Digital Readiness Score (refer to PAS:1040)'

In [6]:
model_name = "facebook/bart-large-mnli"
classifier = pipeline("zero-shot-classification", model=model_name)

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

# Retrieve & wrangle data

In [14]:
data = get_sheet_dfs()
# Comment out the line below to explore only the initial dataset (i.e., firms what applied for at least one grant)
roadmaps_df = pandas.concat([data['Roadmaps'], data['RoadmapsWithoutGrants']])
roadmaps_df = wrangle_roadmaps(roadmaps_df)

/content/digi-inno-road-prod/innoprod/wrangling/wrangling_tools.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  series.loc[mask] = new_value


In [15]:
qual_df = roadmaps_df[['Client ID', drs_col] + qual_cols].copy()
qual_df['Context'] = roadmaps_df.apply(lambda row: ' '.join(filter(None, [row[c] for c in qual_cols])), axis=1)
qual_df = qual_df.drop(columns=qual_cols)
# qual_df
len(qual_df)

821

In [16]:
qual_df = qual_df[qual_df['Context'] != '']
# qual_df
len(qual_df)

517

In [17]:
client_ids = qual_df['Client ID'].to_list()
drs_levels = qual_df[drs_col].to_list()
context = qual_df['Context'].to_list()

In [18]:
questions_prompts = get_sheet_dfs(
    sheet_id="18oFiC2gu6azFaO1XHYFsSpVovBvz08s0hgg97Lu1LGY",
    ranges={"QuestionsPrompts": "Sheet1!A1:G16"}
  )['QuestionsPrompts']
# questions_prompts

In [19]:
candidate_label_cols = [c for c in questions_prompts.columns.to_list() if c.startswith('Level ')]
# candidate_label_cols

# Run pipeline

In [20]:
all_results_df = None

for idx, row in questions_prompts.iterrows():
  hypothesis_template = (row['Question'] + " {}")
  candidate_labels = [row[c] for c in candidate_label_cols]
  results = classifier(
      context,
      candidate_labels=candidate_labels,
      hypothesis_template=hypothesis_template
    )
  for res in results:
    res['Prediction'] = candidate_labels.index(res['labels'][0]) + 1
    res['Probability'] = res['scores'][0]
    res['Confidence'] = abs(math.log(res['scores'][0]) - math.log(res['scores'][1]))
    del res['sequence']
    del res['labels']
    del res['scores']
  results_df = pandas.DataFrame(results)
  results_df['Question'] = row['Question']
  results_df['Client ID'] = client_ids
  results_df['Current DRS'] = drs_levels
  if all_results_df is None:
    all_results_df = results_df
  else:
    all_results_df = pandas.concat([all_results_df, results_df])
  print(f'Completed assessing question "{row['Question']}"')

Completed assessing question "Is the company willing to invest time and resources to introduce new skills or knowledge into its workforce through staff training, new recruitment, or apprenticeship schemes?"
Completed assessing question "Is the management team of the company open to working with academic or public organisations/networks to acquire new knowledge?"
Completed assessing question "Are continuous improvement methods embedded within the culture of the company's management and workforce in relation to technology and digital transformation?"
Completed assessing question "Is the company aware of the possibilities offered by emerging technologies and has it identified commercially-viable opportunities where they could add value to its current business?"
Completed assessing question "Does the company have a clear strategy to utilise advanced technologies to support growth and competitiveness?"
Completed assessing question "Does the company have any highly-specialised skills and kno

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Completed assessing question "Does the company currently utilise any advanced technological capabilities that give it an advantage over competitors?"
Completed assessing question "Is the company able to offer its customers specialised products or services that differentiates it from competitors?"
Completed assessing question "Does the workforce of the company have the appropriate know-how to be able to facilitate the introduction of new technology or processes?"
Completed assessing question "Does the management of the company (especially if it is not the same as the ownership) have sufficient decision making authority to introduce the changes needed for its growth strategy?"
Completed assessing question "Does the company have the flexibility in its structures and processes to introduce new products or services in a way that minimises risk and potential disruption to BAU?"
Completed assessing question "Does the company currently have the resources available to invest in new technologies

# Aggregate and write data

In [21]:
questions_prompts['Question'] = pandas.Series(questions_prompts['Question'], dtype="string")
all_results_df['Question'] = pandas.Series(all_results_df['Question'], dtype="string")
all_results_df = pandas.merge(
    all_results_df,
    questions_prompts[['Question', 'Core', 'Category']],
    on="Question",
    how="left"
  )[[
      'Client ID',
      'Current DRS',
      'Core',
      'Category',
      'Question',
      'Prediction',
      'Probability',
      'Confidence'
    ]]

all_results_df

,Client ID,Current DRS,Core,Category,Question,Prediction,Probability,Confidence
0,65CE53AE-8233-D485-957C-705AF17CCE77,4,Willingness,Workforce,Is the company willing to invest time and reso...,3,0.420895,0.692145
1,00CCFCC3-97CA-83F5-9E32-E34EBB545EE9,6,Willingness,Workforce,Is the company willing to invest time and reso...,3,0.548439,1.195571
2,eb08ecff-5bf9-e60a-9cbf-66e8050adf5a,3,Willingness,Workforce,Is the company willing to invest time and reso...,3,0.747319,1.718721
3,0C163899-5F59-970B-1033-ADCCE41DB525,5,Willingness,Workforce,Is the company willing to invest time and reso...,3,0.515494,0.760543
4,C713DA4C-F07B-F227-351A-5C376EC69358,5,Willingness,Workforce,Is the company willing to invest time and reso...,3,0.570276,1.181092
...,...,...,...,...,...,...,...,...
7750,663cc17e-55ba-4b05-12fa-673337fdd41e,6,Capacity,Customers/Markets,Does the company have long-term key customers ...,2,0.549385,0.643458
7751,7a197a96-e09c-c42b-f6d6-67dd8c4f0370,4,Capacity,Customers/Markets,Does the company have long-term key customers ...,2,0.486530,0.324749
7752,02341D5B-30EF-0018-F1F8-EB52A2784F68,7,Capacity,Customers/Markets,Does the company have long-term key customers ...,2,0.414274,0.558671
7753,b1e6d9f7-3fa0-2be8-16ba-673b2c537acf,<NA>,Capacity,Customers/Markets,Does the company have long-term key customers ...,2,0.451839,0.126304


In [22]:
if IN_COLAB:
  output_dir = os.path.join(dirpath, 'analysis', 'outputs')

all_results_df.to_csv(os.path.join(output_dir, 'BART_WCC_results.csv'), index=False)